# Ch01-03: 이상치 탐지 - 통계적 방법


## 학습 목표

- Mahalanobis 거리의 수학적 원리와 χ² 분포와의 관계를 이해한다
- Grubbs 검정의 가설 구조와 임계값 유도를 수행한다
- Generalized ESD 검정으로 복수 이상치를 탐지한다
- 로버스트 추정량(MCD)을 활용한 다변량 이상치 탐지를 구현한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. Mahalanobis 거리

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})}, \quad D_M^2 \sim \chi^2_p$$

### 2. Grubbs 검정

$$G = \frac{\max_i|x_i-\bar{x}|}{s}, \quad G_{\text{crit}} = \frac{n-1}{\sqrt{n}}\sqrt{\frac{t_{\alpha/(2n),n-2}^2}{n-2+t_{\alpha/(2n),n-2}^2}}$$

### 3. Generalized ESD (Rosner, 1983)

$$R_i = \frac{\max_j|x_j-\bar{x}_i|}{s_i}, \quad \lambda_i = \frac{(n-i)t_{p,n-i-1}}{\sqrt{(n-i-1+t_{p,n-i-1}^2)(n-i+1)}}$$

### 4. MCD (Minimum Covariance Determinant)

$$(\hat{\mu}_{\text{MCD}},\hat{\Sigma}_{\text{MCD}}) = \arg\min_{|J|=h}\det(\mathbf{S}_J)$$

붕괴점 ~50%. 이상치에 의한 평균/공분산 왜곡 방지.


## 구현 가이드

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import mahalanobis

np.random.seed(42)
X_n = np.random.multivariate_normal([0,0], [[1,0.7],[0.7,1]], 200)
X_o = np.random.multivariate_normal([4,4], [[0.3,0],[0,0.3]], 10)
X = np.vstack([X_n, X_o]); labels = np.array([0]*200+[1]*10)

mu = X.mean(axis=0); cov_inv = np.linalg.inv(np.cov(X.T))
D = np.array([mahalanobis(x, mu, cov_inv) for x in X])
thresh = np.sqrt(stats.chi2.ppf(0.975, 2))
outliers = D > thresh

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(X[~outliers,0], X[~outliers,1], c='blue', s=20, alpha=0.6, label='정상')
axes[0].scatter(X[outliers,0], X[outliers,1], c='red', s=40, marker='x', label='이상치')
axes[0].set_title('Mahalanobis 이상치 탐지'); axes[0].legend()
sorted_d2 = np.sort(D**2)
theo = stats.chi2.ppf(np.arange(1,len(sorted_d2)+1)/(len(sorted_d2)+1), 2)
axes[1].scatter(theo, sorted_d2, s=10, alpha=0.5)
axes[1].plot([0,max(theo)],[0,max(theo)],'r--')
axes[1].set_title('χ² QQ-Plot'); plt.tight_layout(); plt.show()


---
## 연습 문제

### 문제 1 [★]

5차원 데이터(n=500, 이상치 20개)에 Mahalanobis 이상치 탐지. χ² QQ-plot과 ROC 곡선으로 성능 평가.

In [ ]:
# TODO: 5차원 데이터 생성
# TODO: Mahalanobis + QQ-plot + ROC


### 문제 2 [★★]

Generalized ESD 검정 직접 구현. 정상+오염 혼합 데이터에서 검정 수행.

> **힌트**: R_i > λ_i인 최대 i가 이상치 수.

In [ ]:
def generalized_esd(data, max_outliers=15, alpha=0.05):
    # TODO
    pass


### 문제 3 [★★]

MCD를 이용한 로버스트 Mahalanobis 구현. 이상치 비율 0~40%에서 표본공분산 vs MCD F1 비교.

In [ ]:
from sklearn.covariance import MinCovDet
# TODO: 비율별 실험


### 문제 4 [★★★]

마스킹/스와핑 효과 시뮬레이션. 클러스터형 이상치에서 Grubbs 실패 예시. 이상치 비율별 Classical vs MCD Precision/Recall. 붕괴점 이론 논의.

In [ ]:
# TODO: 마스킹 효과
# TODO: 비율별 성능
# TODO: 붕괴점 분석


---
## 참고 자료

- Rousseeuw & Van Driessen (1999). Fast MCD. Technometrics.
- Grubbs (1950). Sample Criteria for Outlying Observations.
- Rosner (1983). Generalized ESD. Technometrics.
